In [14]:
import pyEDM
import pandas as pd
import numpy as np
import random
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [3]:
# top 20 features from paper + target variable
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    "Solidity", "texture_uniformity", "texture_smoothness",
    "texture_average_gray_level", "texture_entropy",
    "texture_average_contrast", "H90", "H180", "Hflip",
    "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")
df = df.asfreq("D")

df_filled = df.copy()

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    df_norm[col] = (df_filled[col] - mu) / sigma

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
df_mv = df_mv[["t"] + features]

target = "Lpoly_expected_ml"
predictors = [col for col in features if col != target]

df_mv.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,texture_average_gray_level,texture_entropy,texture_average_contrast,H90,H180,Hflip,Extent,EquivDiameter,ConvexArea,ConvexPerimeter
0,1,0.285287,1.059338,0.967595,0.956625,1.139942,1.226418,0.842266,-1.228078,-0.191169,...,-1.183856,0.480870,-1.005885,-0.901014,0.861091,1.128490,0.102206,1.092577,1.111408,1.118301
1,2,0.065656,1.016480,0.917837,0.908034,1.109145,1.007938,0.563761,-1.247322,0.526462,...,-1.141946,0.640061,-0.878977,-1.001016,0.340277,0.680457,0.656972,1.062576,0.999125,1.020599
2,3,-0.128818,0.671753,0.512658,0.637894,0.852846,0.719154,0.645106,-1.070290,0.486697,...,-1.073251,0.996070,-0.759717,-0.976040,0.078281,0.314316,0.520649,0.807369,0.659696,0.771253
3,4,-0.018094,0.379651,0.165389,0.433051,0.632838,0.501162,0.894757,-0.832234,0.370645,...,-1.029278,1.106370,-0.779254,-0.852571,0.097279,0.420381,0.192816,0.595165,0.375226,0.570793
4,5,0.891676,0.756008,0.593485,0.724495,0.928235,0.917121,0.418304,-1.071257,0.091247,...,-1.184920,0.232139,-0.948879,-0.995896,0.312944,0.654387,0.256093,0.879835,0.773202,0.863842


In [4]:
env = pd.read_csv("../data/environment_all.csv")

# could be relevant but too many missing values for ema imputation to be effective
env = env.drop(columns=[
    'fluorescent_dissolved_organic_matter_eco',
    'sea_water_turbidity_eco',
    'waterlevel_predicted_m',
    'mass_concentration_of_oxygen_in_sea_water_seaphox',
    'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox',
    'fractional_saturation_of_oxygen_in_sea_water_seaphox',
    'sea_water_ph_reported_on_total_scale_seaphox_external'
])

env["date"] = pd.to_datetime(env["date"].astype(str), format="%Y%m%d")
env = env.sort_values("date").set_index("date")
env = env.asfreq("D")

env_features = env.columns.tolist()
env_filled = env.copy()

for col in env_features:
    ema = env[col].ewm(span=30, adjust=False).mean()
    env_filled[col] = env[col].fillna(ema)

df_norm = env_filled.copy()

for col in env_features:
    mu = env_filled[col].mean()
    sigma = env_filled[col].std()
    df_norm[col] = (env_filled[col] - mu) / sigma

df_env = df_norm.copy()
df_env = df_env.reset_index()
df_env["t"] = np.arange(1, len(df_env) + 1)
df_env = df_env[["t"] + env_features]

df_all = pd.merge(df_mv, df_env, on="t", how="inner")

df_env = df_env.merge(df_mv[["t", target]], on="t", how="inner")
df_env.head()

,t,mass_concentration_of_chlorophyll_in_sea_water_ctd,sea_water_electrical_conductivity_ctd,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m,Lpoly_expected_ml
0,1,0.386650,1.505381,0.361222,-0.887083,0.781443,1.526531,-0.299634,0.288179,-0.014762,1.008903,-1.294158,1.006837,0.285287
1,2,0.718488,1.473634,0.375660,-0.833821,0.519131,1.488148,-0.107828,0.303818,-0.095281,1.224096,-0.784883,0.654760,0.065656
2,3,1.065658,0.860840,0.326331,-0.349615,0.544113,0.835239,-0.115066,0.714121,-0.069400,1.365032,-0.344831,0.304790,-0.128818
3,4,1.402899,0.352516,0.254143,0.008697,0.331765,0.303019,-0.799053,-0.682379,-0.785444,1.101345,-0.532718,0.348009,-0.018094
4,5,1.678048,0.440744,0.238502,-0.079670,0.181872,0.407880,-0.665150,0.782198,-0.788319,0.808864,-0.370789,0.219406,0.891676


In [5]:
df_all.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength,MinorAxisLength,Perimeter,Orientation,Eccentricity,Solidity,...,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m
0,1,0.285287,1.059338,0.967595,0.956625,1.139942,1.226418,0.842266,-1.228078,-0.191169,...,0.361222,-0.887083,0.781443,1.526531,-0.299634,0.288179,-0.014762,1.008903,-1.294158,1.006837
1,2,0.065656,1.016480,0.917837,0.908034,1.109145,1.007938,0.563761,-1.247322,0.526462,...,0.375660,-0.833821,0.519131,1.488148,-0.107828,0.303818,-0.095281,1.224096,-0.784883,0.654760
2,3,-0.128818,0.671753,0.512658,0.637894,0.852846,0.719154,0.645106,-1.070290,0.486697,...,0.326331,-0.349615,0.544113,0.835239,-0.115066,0.714121,-0.069400,1.365032,-0.344831,0.304790
3,4,-0.018094,0.379651,0.165389,0.433051,0.632838,0.501162,0.894757,-0.832234,0.370645,...,0.254143,0.008697,0.331765,0.303019,-0.799053,-0.682379,-0.785444,1.101345,-0.532718,0.348009
4,5,0.891676,0.756008,0.593485,0.724495,0.928235,0.917121,0.418304,-1.071257,0.091247,...,0.238502,-0.079670,0.181872,0.407880,-0.665150,0.782198,-0.788319,0.808864,-0.370789,0.219406


https://sugiharalab.github.io/EDM_Documentation/parameters/

In [6]:
def one_simplex(df, target, features, E=4, Tp=1):
    # Randomly select 3 features (+ the target) for the simplex projection
    chosen_features = random.sample(features, 3)
    columns = [target] + chosen_features
    columns_str = " ".join(columns) # has to be 'space separated' idk ????

    N = len(df)
    res = pyEDM.Simplex(
        dataFrame=df,
        columns=columns_str,
        target=target,
        E=E,
        tau=1,
        Tp=Tp,
        lib=f"1 {N}",
        pred=f"1 {N}"
    )

    obs = res["Observations"].to_numpy()
    pred = res["Predictions"].to_numpy()

    mask = np.isfinite(obs) & np.isfinite(pred)
    obs = obs[mask]
    pred = pred[mask]

    if len(obs) < 10 or np.std(obs) == 0 or np.std(pred) == 0:
        return np.nan, chosen_features

    rho = np.corrcoef(obs, pred)[0, 1]
    rmse = np.sqrt(np.mean((obs - pred) ** 2))
    mae = np.mean(np.abs(obs - pred))
    return rho, rmse, mae, chosen_features

def multiview_big(df, target, features, Tp, n_trials=500):
    results = []

    for i in range(n_trials):
        rho, rmse, mae, chosen = one_simplex(df, target, features, E=4, Tp=Tp)

        results.append({
            "rho": rho,
            "rmse": rmse,
            "mae": mae,
            "features": chosen
        })

    return pd.DataFrame(results)

def multiview_yes(df_mv, target, predictors):
    # wrapper main function
    x = df_mv[target].to_numpy()

    summary_rows = []
    feature_importance_by_tp = {}
    
    for Tp in range(1, 32):
        mv = multiview_big(df_mv, target, predictors, Tp, n_trials=500)

        acf = abs(pd.Series(x).autocorr(lag=Tp))
        mv["rho_eff"] = mv["rho"] - acf

        # summary stats
        summary_rows.append({
            "Tp": Tp,
            "rho_eff_mean": mv["rho_eff"].mean(),
            "rmse_mean": mv["rmse"].mean(),
            "mae_mean": mv["mae"].mean(),
            "n_models": len(mv)
        })

        # feature importance (top 5%)
        top = mv.nlargest(int(0.05 * len(mv)), "rho_eff")

        counts = Counter()
        for feats in top["features"]:
            for f in feats:
                counts[f] += 1

        importance = pd.Series(counts) / len(top)
        feature_importance_by_tp[Tp] = importance.sort_values(ascending=False)
    
    return summary_rows, feature_importance_by_tp

In [7]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_mv, target, predictors)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.189712   0.440041  0.083665       500
1    2      0.442287   0.402853  0.079357       500
2    3      0.584600   0.378321  0.080549       500
3    4      0.270328   0.810859  0.181705       500
4    5      0.124765   0.945786  0.217135       500
5    6      0.166202   0.964252  0.225597       500
6    7      0.148709   0.979978  0.238124       500
7    8      0.081443   0.999166  0.240627       500
8    9      0.101150   0.998080  0.240798       500
9   10      0.074567   0.999542  0.244934       500
10  11      0.073577   1.004314  0.248284       500
11  12      0.062171   1.022186  0.264270       500
12  13      0.022963   1.040081  0.277481       500
13  14      0.055528   1.051247  0.284212       500
14  15      0.094359   1.047520  0.288129       500
15  16      0.132465   1.031682  0.293786       500
16  17      0.176095   1.004008  0.291253       500
17  18      0.260152   0.978189  0.295240       500
18  19      

In [8]:
# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)

importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
print(importance_all)

     Tp                     Feature  Proportion
0     1                    Solidity        0.72
1     1               EquivDiameter        0.32
2     1             MajorAxisLength        0.28
3     1                   Biovolume        0.24
4     1                 Orientation        0.24
..   ..                         ...         ...
482  31                         H90        0.12
483  31  texture_average_gray_level        0.08
484  31             ConvexPerimeter        0.08
485  31             MinorAxisLength        0.08
486  31                Eccentricity        0.08

[487 rows x 3 columns]


In [9]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_env, target, env_features)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.179586   0.458111  0.119208       500
1    2      0.434489   0.422448  0.113560       500
2    3      0.575302   0.403382  0.117126       500
3    4      0.269137   0.810223  0.203288       500
4    5      0.133473   0.931285  0.232346       500
5    6      0.191251   0.941640  0.239210       500
6    7      0.197803   0.952743  0.248418       500
7    8      0.111707   0.994093  0.261736       500
8    9      0.131818   0.982198  0.260200       500
9   10      0.118703   0.983147  0.266102       500
10  11      0.141453   0.990412  0.275627       500
11  12      0.203755   0.985428  0.287720       500
12  13      0.225628   0.975554  0.292372       500
13  14      0.273550   0.967709  0.287047       500
14  15      0.264583   0.991812  0.296995       500
15  16      0.238917   0.996799  0.301011       500
16  17      0.239601   0.974212  0.294472       500
17  18      0.201136   1.001725  0.298120       500
18  19      

In [10]:
importance_all = pd.concat(
    feature_importance_by_tp,
    names=["Tp", "Feature"]
).reset_index()

importance_all.columns = ["Tp", "Feature", "Proportion"]
print(importance_all)

     Tp                                            Feature  Proportion
0     1              sea_water_electrical_conductivity_ctd        0.84
1     1                          sea_water_temperature_ctd        0.60
2     1                              sea_water_sigma_t_ctd        0.56
3     1  mass_concentration_of_chlorophyll_in_sea_water...        0.24
4     1                                            baro_mb        0.24
..   ..                                                ...         ...
231  31                   sea_water_practical_salinity_ctd        0.60
232  31  mass_concentration_of_chlorophyll_in_sea_water...        0.52
233  31                              sea_water_sigma_t_ctd        0.44
234  31                          sea_water_temperature_ctd        0.32
235  31              sea_water_electrical_conductivity_ctd        0.28

[236 rows x 3 columns]


In [11]:
summary_rows = []
feature_importance_by_tp = {}

summary_rows, feature_importance_by_tp = multiview_yes(df_all, target, predictors+env_features)
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

    Tp  rho_eff_mean  rmse_mean  mae_mean  n_models
0    1      0.186429   0.447181  0.095394       500
1    2      0.439702   0.409002  0.089242       500
2    3      0.580869   0.387907  0.092228       500
3    4      0.265278   0.814490  0.189349       500
4    5      0.125789   0.942908  0.220727       500
5    6      0.174119   0.958740  0.228539       500
6    7      0.173159   0.967002  0.236849       500
7    8      0.101363   0.992194  0.242191       500
8    9      0.127742   0.983642  0.239358       500
9   10      0.099643   0.986588  0.243372       500
10  11      0.101918   0.995582  0.250016       500
11  12      0.133526   0.998013  0.263170       500
12  13      0.114534   1.006835  0.274097       500
13  14      0.169262   1.004272  0.275604       500
14  15      0.193734   1.013687  0.283802       500
15  16      0.191164   1.016584  0.291810       500
16  17      0.221260   0.990652  0.289401       500
17  18      0.229215   1.000039  0.294520       500
18  19      

In [17]:
import plotly.express as px
import plotly.graph_objects as go

# 1. rho_eff over Tp
px.line(summary_df, x="Tp", y="rho_eff_mean", title="ρ_eff over Tp", markers=True).show()

# 2. RMSE & MAE over Tp
fig = go.Figure()
fig.add_scatter(x=summary_df["Tp"], y=summary_df["rmse_mean"], name="RMSE", mode="lines+markers")
fig.add_scatter(x=summary_df["Tp"], y=summary_df["mae_mean"],  name="MAE",  mode="lines+markers")
fig.update_layout(title="RMSE & MAE over Tp", xaxis_title="Tp", yaxis_title="Error")
fig.show()

# 3. Feature importance heatmap
pivot = importance_all.pivot_table(index="Feature", columns="Tp", values="Proportion", aggfunc="mean")
px.imshow(pivot, aspect="auto", color_continuous_scale="Viridis", title="Feature Importance by Tp").show()

# 4. Top features (mean across all Tp)
mean_imp = importance_all.groupby("Feature")["Proportion"].mean().sort_values().tail(15)
px.bar(mean_imp, orientation="h", title="Mean Feature Importance").show()